# CrossHateTN : protocole Bronze complet

Ce notebook refait tout le protocole Bronze depuis les sources :

- données auditées et split gelé ;
- annotation pilote et kappa ;
- règles : normalisation, lexique, translittération Arabizi ;
- TF-IDF + SVM, choisi sur la validation ;
- évaluation **unique** sur le test gelé ;
- scores par écriture ;
- faux positifs sur insultes et citations ;
- 30 erreurs catégorisées ;
- reproduction d'un résultat publié ;
- tableaux 1 à 3 et `experiments.md`.

**Entrées à ajouter (Add Input)**
1. Le dépôt : `crosshatetn_bronze.zip`, ajouté comme dataset.
2. Le fichier TUNIZI annoté par l'équipe : `TUNIZI_annotated.xlsx` (et, pour le kappa entre deux annotations, l'ancien `TUNIZI-Dataset_annote_arabe.csv`).
3. Les annotations remplies : `annotation_kappa_tunizi.xlsx` et `annotations_equipe.xlsx`.

**Session** : activez Internet, car les sources sont téléchargées puis vérifiées par SHA-256. Le CPU suffit.

**Ordre d'exécution**
1. *Run All* avec `EVALUER_TEST = False` : tout est calculé, sauf le test.
2. Vérifiez les sorties : modèle retenu, lexique, kappa, erreurs.
3. Passez `EVALUER_TEST = True` et faites *Run All* **une seule fois**.

Aucun choix ne dépend du test. La configuration est retenue automatiquement sur la validation, et chaque évaluation du test est journalisée.

Les sorties vont dans `/kaggle/working/bronze_outputs/`, avec une archive `bronze_outputs.zip` à télécharger et à versionner dans le dépôt.

In [ ]:
# ---- Réglages du protocole : ne plus les modifier après avoir vu le test ----
EVALUER_TEST = False      # True pour l'évaluation finale sur le test gelé, une seule fois
INCLURE_LHSAB = True      # variante « + L-HSAB », mesurée sur la validation uniquement
SEED = 42
N_BOOT = 1000
SEUIL_FUITE = 0.70        # même seuil que les quasi-doublons du dépôt (configs/data.yaml)
N_ERREURS = 30
# True seulement si les deux fichiers TUNIZI complets ont été annotés par deux personnes différentes,
# chacune sans voir les étiquettes de l'autre : le kappa est alors calculé sur toutes les phrases communes.
ANNOTATIONS_INDEPENDANTES = True

# Lot responsable de chaque composant : à adapter à la répartition réelle de l'équipe
LOT = {"Données et split": "A", "Annotation pilote": "A", "Normalisation (règles)": "B",
       "Lexique (règles)": "B", "Translittération (règles)": "B", "TF-IDF + SVM": "C",
       "Sortie et démo": "E", "Reproduction": "Chef de projet"}

SPLIT_FINAL = "test"      # ne pas modifier : split T-HSAB de l'évaluation finale
TUNIZI_FINAL = "test"     # ne pas modifier : moitié TUNIZI de l'évaluation finale

In [ ]:
import os, sys, re, json, time, shutil, hashlib, platform, inspect, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

KAGGLE = Path("/kaggle/input").exists()
INPUT = Path("/kaggle/input") if KAGGLE else Path(os.environ.get("CROSSHATETN_INPUT", "entrees")).resolve()
WORK = Path("/kaggle/working") if KAGGLE else Path(os.environ.get("CROSSHATETN_WORK", "travail")).resolve()
OUT = WORK / "bronze_outputs"
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)


def trouver_depot(racine: Path) -> Path:
    for p in sorted(racine.rglob("pyproject.toml")):
        if 'name = "crosshatetn"' in p.read_text(encoding="utf-8", errors="ignore"):
            return p.parent
    raise FileNotFoundError("Dépôt crosshatetn introuvable : ajoutez le zip du dépôt en entrée (Add Input).")


SOURCE_DEPOT = trouver_depot(INPUT)
REPO = WORK / "crosshatetn"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SOURCE_DEPOT, REPO, ignore=shutil.ignore_patterns("__pycache__", ".git", "outputs"))
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

# Sources brutes déjà fournies en entrée (facultatif) : copiées à l'endroit attendu, SHA-256 vérifié ensuite
for nom, dossier in [("T-HSAB.xlsx", "thsab"), ("L-HSAB", "lhsab"), ("TUNIZI-Dataset.txt", "tunizi")]:
    trouves = [p for p in INPUT.rglob(nom) if p.is_file() and SOURCE_DEPOT not in p.parents]
    if trouves:
        (REPO / "data" / "raw" / dossier).mkdir(parents=True, exist_ok=True)
        shutil.copy(trouves[0], REPO / "data" / "raw" / dossier / nom)


def lire(p: Path):
    if p.suffix.lower() == ".xlsx":
        return list(pd.read_excel(p, sheet_name=None, dtype=str).items())
    return [(p.stem, pd.read_csv(p, dtype=str, keep_default_na=False))]


TABLES = {}
for p in sorted(INPUT.rglob("*")):
    if (p.is_file() and p.suffix.lower() in (".csv", ".xlsx") and SOURCE_DEPOT not in p.parents
            and p.name != "T-HSAB.xlsx" and p.stat().st_size < 50_000_000):
        try:
            for feuille, df in lire(p):
                df.columns = [str(c).strip() for c in df.columns]
                TABLES[f"{p.name} > {feuille}"] = df.fillna("").astype(str)
        except Exception as exc:
            print("Lecture impossible :", p.name, "-", exc)


def trouver(colonnes: set, quoi: str, obligatoire: bool = False):
    for nom, df in TABLES.items():
        if colonnes <= set(df.columns):
            print(f"{quoi} : {nom} ({len(df)} lignes)")
            return df.copy()
    if obligatoire:
        raise FileNotFoundError(f"{quoi} introuvable (colonnes attendues : {sorted(colonnes)})")
    print(f"{quoi} : non fourni")
    return None


def normaliser_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


TUNIZI_EQUIPE = trouver({"id", "label", "text_arabizi", "text_arabic"}, "TUNIZI annoté (fichier de l'équipe)")
TUNIZI_ANCIEN = trouver({"id", "label", "text", "text_arabe"}, "TUNIZI annoté (ancien fichier)")
if TUNIZI_EQUIPE is not None:
    TUNIZI_ANNOTE = TUNIZI_EQUIPE.rename(columns={"text_arabizi": "text", "text_arabic": "text_arabe"})
elif TUNIZI_ANCIEN is not None:
    TUNIZI_ANNOTE = TUNIZI_ANCIEN
else:
    raise FileNotFoundError("Fichier TUNIZI annoté introuvable : ajoutez-le en entrée (Add Input).")
print("Annotation TUNIZI utilisée pour l'évaluation :", "fichier de l'équipe" if TUNIZI_EQUIPE is not None else "ancien fichier")
KAPPA = trouver({"id", "texte", "label_annotateur_2"}, "Double annotation (kappa)")
LEXIQUE = trouver({"terme", "garder"}, "Lexique relu")
CONTRASTE = trouver({"phrase", "label", "label_propose"}, "Jeu de contraste")
ERREURS = trouver({"id", "categorie_validee"}, "30 erreurs relues")

RES = {}  # une ligne de experiments.md par clé


def noter(cle, composant="", configuration="", metrique="", dev=None, test=None, autre=None, source=""):
    ligne = RES.setdefault(cle, {"composant": composant, "configuration": configuration, "métrique": metrique,
                                 "dev": None, "test": None, "autre": None, "source": source})
    for champ, valeur in (("dev", dev), ("test", test), ("autre", autre)):
        if valeur is not None:
            ligne[champ] = valeur
    ligne["date"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


print("Python", platform.python_version(), "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
print("Dépôt :", SOURCE_DEPOT)

## A. Données : sources figées, nettoyage, split gelé, audit

Le pipeline du dépôt télécharge chaque source à un commit précis et vérifie son SHA-256. Il écarte les doublons et les conflits d'étiquettes, groupe les quasi-doublons, recharge le split gelé et contrôle les fuites. Le notebook s'arrête si les données ne correspondent plus exactement au gel.

In [ ]:
from crosshatetn.config import load_config
from crosshatetn.pipeline import prepare
from crosshatetn.splits import ids_digest, load_manifest

cfg = load_config("configs/data.yaml")
prep = prepare(cfg, write=False, verbose=True)
lab = prep.labeled
so = prep.split_outcome
assert not so.created, "Aucun split gelé : le dossier splits/ manque dans le dépôt."
assert len(so.added) == 0, (f"{len(so.added)} texte(s) absent(s) du gel : les données ne sont plus celles du gel. "
                            "Vérifier annotations/arbitrage.csv et configs/data.yaml, ne pas continuer.")
assert len(so.stale_ids) == 0, f"{len(so.stale_ids)} identifiant(s) gelé(s) absent(s) des données actuelles."
assert prep.audit.ok, "Audit des fuites en échec : ne pas continuer."

th = lab[lab.source == "thsab"]
EFFECTIFS = th.split.value_counts().to_dict()
assert EFFECTIFS == {"train": 4164, "validation": 894, "test": 893}, f"Effectifs T-HSAB inattendus : {EFFECTIFS}"

manifest = load_manifest(cfg.path("splits"))
DATE_GEL = manifest["historique"][0]["date"]
HASH_TEST = manifest["digests"]["test"]
assert ids_digest(lab.loc[lab.split == "test", "id"]) == HASH_TEST
print(f"Preuve du gel : splits créés le {DATE_GEL} ; SHA-256 des identifiants du test : {HASH_TEST}")

resume, repartition, ecritures, audit = prep.summary(), prep.split_table(), prep.script_table(), prep.audit.table()
for nom, table in [("donnees_resume", resume), ("donnees_repartition", repartition),
                   ("donnees_ecritures", ecritures), ("donnees_audit", audit)]:
    table.to_csv(OUT / f"{nom}.csv")
    display(table)

etats = audit["résultat"].value_counts().to_dict()
noter("gel", "Données et split", "StratifiedGroupKFold, 20 plis, graine 42 ; quasi-doublons ≥ 0,70 dans le même split",
      "date du gel ; SHA-256 du test", autre=f"{DATE_GEL} ; {HASH_TEST[:16]}…", source="section A")
noter("effectifs", "Données et split", "T-HSAB après nettoyage", "train / validation / test",
      autre="4164 / 894 / 893", source="section A")
noter("audit", "Données et split", "contrôles de fuite entre splits", "résultat",
      autre=" ; ".join(f"{k} : {v}" for k, v in etats.items()), source="section A")

In [ ]:
# Carte de données du dépôt : source, licence, données personnelles
print((REPO / "docs" / "carte_donnees.md").read_text(encoding="utf-8")[:3000])

## B. Annotation pilote TUNIZI et contrôle des fuites par translittération

TUNIZI a été réannoté par l'équipe. Le kappa se calcule sur une double annotation **aveugle** de phrases tirées au hasard : le second annotateur ne voit pas les premières étiquettes.

Contrôle de fuite propre à P9 : une phrase TUNIZI dont la translittération est un quasi-doublon (≥ 0,70) d'un commentaire d'entraînement T-HSAB est retirée de l'évaluation Arabizi. C'est la règle des quasi-doublons du dépôt.

TUNIZI est ensuite coupé en deux moitiés fixes (graine 42) :
- **dev** : vérifier les règles de translittération ;
- **test** : mesure finale uniquement.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from crosshatetn.text import dedup_key, normalize_light, normalize_strong

B2 = {"normal": "non_haineux", "abusive": "haineux", "hate": "haineux"}
L2 = ["non_haineux", "haineux"]
ann = TUNIZI_ANNOTE.copy()
ann["id"] = normaliser_id(ann["id"])
ann["label"] = ann["label"].str.strip().str.lower()
a_verifier = int((ann["a_verifier"].str.strip().str.lower() == "oui").sum()) if "a_verifier" in ann else 0
print("TUNIZI annoté :", len(ann), "phrases ;", ann.label.value_counts().to_dict(), "; marquées à vérifier :", a_verifier)
MOJIBAKE = re.compile(r"Ã|Â|â€")
propre = ann[~ann.text.str.contains(MOJIBAKE)]
propre = propre.assign(cle=propre.text.map(dedup_key)).drop_duplicates("cle")
print("Sans textes mal encodés ni doublons :", len(propre), propre.label.value_counts().to_dict())
noter("tunizi_annotation", "Annotation pilote", "TUNIZI réannoté par l'équipe (3 000 phrases au départ)",
      "étiquettes après nettoyage", source="section B",
      autre=" / ".join(f"{k} {v}" for k, v in propre.label.value_counts().items()) + f" ; à vérifier : {a_verifier}")

print((REPO / "docs" / "guide_annotation.md").read_text(encoding="utf-8")[:1500])

KAPPA_RES = None
if KAPPA is not None:
    k = KAPPA.assign(id=normaliser_id(KAPPA["id"]), label_annotateur_2=KAPPA["label_annotateur_2"].str.strip().str.lower())
    k = k[k.label_annotateur_2.isin(["normal", "abusive", "hate"])].merge(ann[["id", "label"]], on="id", how="inner")
    if len(k):
        KAPPA_RES = {"n": len(k), "kappa_3": cohen_kappa_score(k.label, k.label_annotateur_2),
                     "kappa_2": cohen_kappa_score(k.label.map(B2), k.label_annotateur_2.map(B2)),
                     "accord": float((k.label == k.label_annotateur_2).mean())}
        print(f"Kappa de Cohen sur {KAPPA_RES['n']} phrases : 3 étiquettes = {KAPPA_RES['kappa_3']:.3f} ; "
              f"2 classes = {KAPPA_RES['kappa_2']:.3f} ; accord brut = {KAPPA_RES['accord']:.1%}")
        display(pd.crosstab(k.label.rename("annotateur 1"), k.label_annotateur_2.rename("annotateur 2")))
        k[["id", "label", "label_annotateur_2"]].to_csv(OUT / "double_annotation_tunizi.csv", index=False)
        config_kappa = f"double annotation aveugle de {len(k)} phrases tirées au hasard (graine 42)"
        noter("kappa2", "Annotation pilote", config_kappa, "kappa de Cohen (2 classes)", autre=KAPPA_RES["kappa_2"], source="section B")
        noter("kappa3", "Annotation pilote", config_kappa, "kappa de Cohen (3 étiquettes)", autre=KAPPA_RES["kappa_3"], source="section B")
if KAPPA_RES is None and ANNOTATIONS_INDEPENDANTES and TUNIZI_EQUIPE is not None and TUNIZI_ANCIEN is not None:
    k = (TUNIZI_ANCIEN.assign(id=normaliser_id(TUNIZI_ANCIEN["id"]), label_1=TUNIZI_ANCIEN["label"].str.strip().str.lower())[["id", "label_1"]]
         .merge(TUNIZI_EQUIPE.assign(id=normaliser_id(TUNIZI_EQUIPE["id"]), label_2=TUNIZI_EQUIPE["label"].str.strip().str.lower())[["id", "label_2"]], on="id"))
    k = k[k.label_1.isin(["normal", "abusive", "hate"]) & k.label_2.isin(["normal", "abusive", "hate"])]
    KAPPA_RES = {"n": len(k), "kappa_3": cohen_kappa_score(k.label_1, k.label_2),
                 "kappa_2": cohen_kappa_score(k.label_1.map(B2), k.label_2.map(B2)), "accord": float((k.label_1 == k.label_2).mean())}
    print(f"Kappa de Cohen entre les deux annotations complètes ({KAPPA_RES['n']} phrases) : 3 étiquettes = {KAPPA_RES['kappa_3']:.3f} ; "
          f"2 classes = {KAPPA_RES['kappa_2']:.3f} ; accord brut = {KAPPA_RES['accord']:.1%}")
    display(pd.crosstab(k.label_1.rename("annotation 1"), k.label_2.rename("annotation 2")))
    m2 = pd.crosstab(k.label_1.map(B2), k.label_2.map(B2))
    print("Tableau 2 classes (lignes : annotation 1, colonnes : annotation 2) :")
    display(m2)
    k.to_csv(OUT / "double_annotation_tunizi.csv", index=False)
    config_kappa = f"deux annotations complètes et indépendantes de TUNIZI ({len(k)} phrases)"
    noter("kappa2", "Annotation pilote", config_kappa, "kappa de Cohen (2 classes)", autre=KAPPA_RES["kappa_2"], source="section B")
    noter("kappa3", "Annotation pilote", config_kappa, "kappa de Cohen (3 étiquettes)", autre=KAPPA_RES["kappa_3"], source="section B")
    noter("kappa_matrice", "Annotation pilote", config_kappa,
          "2 classes : haineux/haineux ; haineux/non ; non/haineux ; non/non (annotation 1/annotation 2)",
          autre=" ; ".join(str(int(m2.loc[a, b])) for a in ["haineux", "non_haineux"] for b in ["haineux", "non_haineux"]),
          source="section B")
if KAPPA_RES is None:
    print("Kappa non calculé : le fichier de double annotation est absent ou vide.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from crosshatetn.translit import translitterer

ev = propre[propre.label.isin(["normal", "abusive"])].copy()   # hate : trop rare dans TUNIZI pour être évalué seul
ev["y"] = ev.label.map(B2)
ev["moitie"] = ev["id"].map(lambda i: "dev" if int(hashlib.sha256(f"{SEED}:tunizi:{i}".encode()).hexdigest()[:12], 16)
                            / 16 ** 12 < 0.5 else "test")
ev["arabe_regles"] = ev.text.map(translitterer)


def sim_max(requetes, reference) -> np.ndarray:
    # Similarité cosinus maximale de chaque requête avec la référence (n-grammes de caractères 3-5, comme le dépôt)
    requetes, reference = list(requetes), list(reference)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, dtype=np.float32)
    vec.fit(reference + requetes)
    R, Q = vec.transform(reference), vec.transform(requetes)
    out = np.zeros(Q.shape[0], dtype=np.float32)
    for s in range(0, Q.shape[0], 500):
        out[s:s + 500] = (Q[s:s + 500] @ R.T).max(axis=1).toarray().ravel()
    return out


ref_train = th[th.split == "train"].text_raw.map(normalize_light)
s_train = np.maximum(sim_max(ev.text_arabe.map(normalize_light), ref_train),
                     sim_max(ev.arabe_regles.map(normalize_light), ref_train))
ev["fuite_train"] = s_train >= SEUIL_FUITE
dev_masque = (ev.moitie == "dev").to_numpy()
proche = np.zeros(len(ev), dtype=bool)
s_dev = np.maximum(sim_max(ev.text[~dev_masque].str.lower(), ev.text[dev_masque].str.lower()),
                   sim_max(ev.text_arabe[~dev_masque].map(normalize_light), ev.text_arabe[dev_masque].map(normalize_light)))
proche[~dev_masque] = s_dev >= SEUIL_FUITE
ev["proche_dev"] = proche
fuite = ev.fuite_train.to_numpy()
TZ = {"dev": ev[dev_masque & ~fuite], "test": ev[~dev_masque & ~fuite & ~proche]}
print(f"TUNIZI évaluable : {len(ev)} phrases ; quasi-doublons d'un commentaire d'entraînement T-HSAB : {int(fuite.sum())} (exclus)")
print(f"TUNIZI-test : {int(proche.sum())} phrase(s) proche(s) d'une phrase de TUNIZI-dev (exclues)")
for moitie, d in TZ.items():
    print(f"TUNIZI-{moitie} : {len(d)} phrases {d.y.value_counts().to_dict()}")
display(ev[fuite].assign(similarite=s_train[fuite].round(2))[["text", "text_arabe", "similarite"]].head(10))
noter("tunizi_fuite", "Données et split",
      "TUNIZI : quasi-doublon (≥ 0,70) d'un commentaire d'entraînement T-HSAB après translittération",
      "phrases exclues de l'évaluation Arabizi", autre=int(fuite.sum()), source="section B")
noter("tunizi_moities", "Données et split", "TUNIZI en deux moitiés (graine 42) : dev pour vérifier, test pour mesurer",
      "phrases dev / test", autre=f"{len(TZ['dev'])} / {len(TZ['test'])}", source="section B")

## C. TF-IDF + SVM et normalisation, choisis sur la validation

Grille complète sur la validation T-HSAB (894 commentaires) :

- représentation : mots (1 à 2 mots), caractères (2 à 5, `char_wb`) ou les deux ;
- texte : brut, normalisation légère ou forte ;
- C dans {0,03 ; 0,1 ; 0,3 ; 1 ; 3}.

La famille du modèle est fixée a priori : SVM linéaire avec classes équilibrées. La configuration retenue est celle qui donne la meilleure macro-F1 de validation, et le plus petit C en cas d'égalité. La régression logistique n'est montrée que pour comparaison.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_union
from sklearn.svm import LinearSVC
from crosshatetn.metrics import bootstrap_ci, confusion_table, paired_bootstrap, scores

tr, va = th[th.split == "train"], th[th.split == "validation"]
y_tr, y_va = tr.label.map(B2).to_numpy(), va.label.map(B2).to_numpy()
NORMALISER = {"text_raw": lambda t: t, "text_light": normalize_light, "text_strong": normalize_strong}


def representation(nom: str):
    mots = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True)
    car = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    return {"mots": mots, "caracteres": car, "mots+caracteres": make_union(mots, car)}[nom]


GRILLE_C = [0.03, 0.1, 0.3, 1.0, 3.0]
lignes, PRED_VA, MODELES = [], {}, {}
for rep in ["mots", "caracteres", "mots+caracteres"]:
    for variante in ["text_raw", "text_light", "text_strong"]:
        f = representation(rep).fit(tr[variante])
        X_tr, X_va = f.transform(tr[variante]), f.transform(va[variante])
        for C in GRILLE_C:
            m = LinearSVC(C=C, class_weight="balanced", random_state=SEED).fit(X_tr, y_tr)
            cle = ("svm", rep, variante, C)
            PRED_VA[cle], MODELES[cle] = m.predict(X_va), (f, m)
            lignes.append({"modele": "svm", "representation": rep, "texte": variante, "C": C,
                           **scores(y_va, PRED_VA[cle], L2)})
for rep in ["caracteres", "mots+caracteres"]:  # comparaison seulement
    f = MODELES[("svm", rep, "text_raw", 1.0)][0]
    X_tr, X_va = f.transform(tr.text_raw), f.transform(va.text_raw)
    for C in [0.3, 1.0, 3.0, 10.0, 30.0]:
        p = LogisticRegression(C=C, class_weight="balanced", max_iter=5000, random_state=SEED).fit(X_tr, y_tr).predict(X_va)
        lignes.append({"modele": "regression_logistique", "representation": rep, "texte": "text_raw", "C": C,
                       **scores(y_va, p, L2)})

GRILLE = pd.DataFrame(lignes)
GRILLE.round(4).to_csv(OUT / "grille_validation.csv", index=False)
grille_svm = GRILLE[GRILLE.modele == "svm"].sort_values(["macro_f1", "C"], ascending=[False, True], kind="mergesort")
MEILLEURS = grille_svm.drop_duplicates(["representation", "texte"])
REF = grille_svm.iloc[0]
K_REF = ("svm", REF.representation, REF.texte, float(REF.C))
f_ref, m_ref = MODELES[K_REF]
print(f"Configuration retenue sur la validation : {REF.representation}, {REF.texte}, C = {REF.C:g} "
      f"(macro-F1 {REF.macro_f1:.4f})")
if K_REF[1:] != ("caracteres", "text_raw", 1.0):
    print("ATTENTION : la configuration retenue diffère de celle de la démo et des diapos "
          "(caractères, texte brut, C = 1). Mettre la démo et les diapos à jour.")
display(MEILLEURS[["representation", "texte", "C", "macro_f1", "accuracy", "f1_haineux", "recall_haineux"]].round(4))
logreg = GRILLE[GRILLE.modele == "regression_logistique"].sort_values("macro_f1", ascending=False).drop_duplicates("representation")
display(logreg[["representation", "C", "macro_f1"]].round(4))
for _, r in logreg.iterrows():
    noter(f"logreg::{r.representation}", "TF-IDF + SVM", f"comparaison : régression logistique, {r.representation}, C = {r.C:g}",
          "macro-F1 T-HSAB (2 classes)", dev=r.macro_f1, source="section C")

In [ ]:
def cle_meilleure(rep, variante="text_raw"):
    r = MEILLEURS[(MEILLEURS.representation == rep) & (MEILLEURS.texte == variante)].iloc[0]
    return ("svm", rep, variante, float(r.C))


NOM_REF = "TF-IDF caractères + SVM (référence)" if K_REF[1] == "caracteres" else f"TF-IDF {K_REF[1]} + SVM (référence)"
SYSTEMES = {"Majoritaire": None, "TF-IDF mots + SVM": cle_meilleure("mots"),
            "TF-IDF mots + caractères + SVM": cle_meilleure("mots+caracteres"), NOM_REF: K_REF}
NOM_BRUT = NOM_REF
if K_REF[2] != "text_raw":
    NOM_BRUT = "Référence, texte brut"
    SYSTEMES[NOM_BRUT] = cle_meilleure(K_REF[1], "text_raw")
for variante, nom in [("text_light", "Normalisation légère"), ("text_strong", "Normalisation forte")]:
    if K_REF[2] != variante:
        SYSTEMES[nom] = cle_meilleure(K_REF[1], variante)
COMPOSANT = {nom: ("Normalisation (règles)" if nom.startswith("Normalisation") else
                   "Baseline triviale" if cle is None else "TF-IDF + SVM") for nom, cle in SYSTEMES.items()}


def predire_cle(cle, donnees):
    if cle is None:
        return np.array(["non_haineux"] * len(donnees))
    f, m = MODELES[cle]
    return m.predict(f.transform(donnees[cle[2]]))


def predire_ref(textes):
    norm = NORMALISER[K_REF[2]]
    return m_ref.predict(f_ref.transform([norm(t) for t in textes]))


def score_ref(textes):
    norm = NORMALISER[K_REF[2]]
    d = m_ref.decision_function(f_ref.transform([norm(t) for t in textes]))
    return d if m_ref.classes_[1] == "haineux" else -d


COLONNES = ["accuracy", "precision_haineux", "recall_haineux", "f1_haineux", "macro_f1"]
lignes = []
for nom, cle in SYSTEMES.items():
    p = predire_cle(cle, va) if cle is None else PRED_VA[cle]
    s = scores(y_va, p, L2)
    config = "toujours « non haineux »" if cle is None else f"{cle[1]}, {cle[2]}, C = {cle[3]:g}"
    lignes.append({"système": nom, "configuration": config, **{k: s[k] for k in COLONNES}})
    noter(f"thsab::{nom}", COMPOSANT[nom], config, "macro-F1 T-HSAB (2 classes)", dev=s["macro_f1"], source="section C")
RESULTATS_DEV = pd.DataFrame(lignes)
RESULTATS_DEV.round(4).to_csv(OUT / "resultats_validation.csv", index=False)
display(RESULTATS_DEV.round(3))

s = scores(y_va, PRED_VA[K_REF], L2)
pt, lo, hi = bootstrap_ci(y_va, PRED_VA[K_REF], L2, N_BOOT, SEED)
noter("thsab::ref_f1h", "TF-IDF + SVM", NOM_REF, "F1 haineux / rappel haineux", dev=f"{s['f1_haineux']:.3f} / {s['recall_haineux']:.3f}", source="section C")
noter("thsab::ref_ic", "TF-IDF + SVM", NOM_REF, "IC 95 % de la macro-F1 (bootstrap, 1 000 tirages)", dev=f"[{lo:.3f} ; {hi:.3f}]", source="section C")
conf_dev = confusion_table(y_va, PRED_VA[K_REF], L2)
conf_dev.to_csv(OUT / "confusion_validation.csv")
display(conf_dev)

lignes = []
for nom, cle in SYSTEMES.items():
    if cle == K_REF:
        continue
    autre = predire_cle(cle, va) if cle is None else PRED_VA[cle]
    c = paired_bootstrap(y_va, PRED_VA[K_REF], autre, L2, N_BOOT, SEED)
    lignes.append({"référence contre": nom, **c})
    noter(f"cmp::{nom}", COMPOSANT[nom], f"{NOM_REF} − {nom}", "écart de macro-F1 [IC 95 %] ; p (bootstrap apparié)",
          dev=f"{c['diff_macro_f1']:+.3f} [{c['ic_bas']:+.3f} ; {c['ic_haut']:+.3f}] ; p = {c['p_value']:.3f}", source="section C")
display(pd.DataFrame(lignes).round(4))

## D. Lexique (règles)

Les candidats viennent **seulement de l'entraînement** T-HSAB : ce sont les mots plus fréquents dans les messages haineux. L'équipe a gardé à la main les vrais termes insultants ou haineux, dans la feuille `lexique` de `annotations_equipe.xlsx`.

La règle signale un message haineux s'il contient un terme gardé. Le terme peut être un mot entier ou un mot précédé d'un préfixe collé (و، ال، ب، ل، ف، يا).

In [ ]:
from crosshatetn import lexique as lx

CANDIDATS = lx.candidats(tr.text_raw, list(tr.label.map(B2)), min_haineux=5, k=200)
CANDIDATS.round(3).to_csv(OUT / "lexique_candidats.csv", index=False)
LEX = set()
if LEXIQUE is not None:
    garde = LEXIQUE[LEXIQUE.garder.str.strip().str.lower().isin(["oui", "1", "x", "yes", "true", "o"])]
    LEX = lx.charger(garde.terme)
if LEX:
    print(len(LEX), "termes gardés sur", len(LEXIQUE), "candidats relus")
    pd.Series(sorted(LEX), name="terme").to_csv(OUT / "lexique_valide.csv", index=False)
    P_LEX_VA = np.array([lx.predire(t, LEX) for t in va.text_raw])
    s = scores(y_va, P_LEX_VA, L2)
    config_lex = f"{len(LEX)} termes gardés à la main ; préfixes collés retirés"
    noter("thsab::lexique", "Lexique (règles)", config_lex, "macro-F1 T-HSAB (2 classes)", dev=s["macro_f1"], source="section D")
    noter("thsab::lexique_rappel", "Lexique (règles)", config_lex, "précision / rappel haineux",
          dev=f"{s['precision_haineux']:.3f} / {s['recall_haineux']:.3f}", source="section D")
    print(f"Lexique sur la validation : macro-F1 = {s['macro_f1']:.4f} ; précision haineux = {s['precision_haineux']:.3f} ; "
          f"rappel haineux = {s['recall_haineux']:.3f}")
    trace = va.assign(termes=va.text_raw.map(lambda t: lx.termes_trouves(t, LEX)))
    display(trace[trace.termes.map(len) > 0][["text_raw", "label", "termes"]].head(5))
    # Faux positifs sur insultes : messages non haineux qui contiennent un terme du lexique
    insulte_nh = (y_va == "non_haineux") & trace.termes.map(len).gt(0).to_numpy()
    fp = int((PRED_VA[K_REF][insulte_nh] == "haineux").sum())
    noter("fp_insultes", "TF-IDF + SVM", "messages non haineux contenant un terme du lexique",
          "faux positifs du SVM de référence", dev=f"{fp} / {int(insulte_nh.sum())}", source="section D")
    print(f"Messages non haineux contenant un terme du lexique : {int(insulte_nh.sum())} ; signalés à tort par le SVM : {fp}")
else:
    print("Lexique non fourni ou vide : règle non évaluée. Les candidats sont dans bronze_outputs/lexique_candidats.csv.")

## E. Translittération Arabizi → arabe par règles

Les règles sont dans `src/crosshatetn/translit.py`. Elles ont été écrites **a priori**, à partir des conventions de l'Arabizi : chiffres, digraphes, voyelles brèves non écrites. Elles n'ont pas été modifiées après leur vérification sur TUNIZI-dev.

Trois entrées sont comparées, avec le SVM de référence :
- l'Arabizi brut, qui sert de baseline « copie de la source » ;
- la translittération par règles ;
- la translittération faite par l'équipe.

In [ ]:
from crosshatetn import translit as tl

print("Chiffres :", tl.DIGITS)
print("Digraphes :", dict(tl.DIGRAPHS))
VUES_TZ = {"Arabizi brut (copie de la source)": "text", "Translittération par règles": "arabe_regles",
           "Translittération de référence (équipe)": "text_arabe"}
d = TZ["dev"]
for nom, col in VUES_TZ.items():
    s = scores(d.y.to_numpy(), predire_ref(d[col]), L2)
    noter(f"tunizi::{nom}", "Translittération (règles)", nom + " → SVM de référence", "macro-F1 TUNIZI (2 classes)",
          dev=s["macro_f1"], source="section E")
    print(f"{nom:42s} macro-F1 TUNIZI-dev = {s['macro_f1']:.4f}")
display(d[["text", "arabe_regles", "text_arabe"]].head(8))

## F. Variante + L-HSAB, mesurée sur la validation seulement

On ajoute le train de L-HSAB (levantin) à l'entraînement, avec la même représentation et la même grille de C. Le modèle de référence reste inchangé : cette variante n'est pas évaluée sur le test.

In [ ]:
if INCLURE_LHSAB:
    train_lev = lab[lab.source.isin(["thsab", "lhsab"]) & (lab.split == "train")]
    f = representation(K_REF[1]).fit(train_lev[K_REF[2]])
    X_lev, X_va = f.transform(train_lev[K_REF[2]]), f.transform(va[K_REF[2]])
    essais = []
    for C in GRILLE_C:
        p = LinearSVC(C=C, class_weight="balanced", random_state=SEED).fit(X_lev, train_lev.label.map(B2)).predict(X_va)
        essais.append((scores(y_va, p, L2)["macro_f1"], -C, C, p))
    mf_lev, _, C_lev, p_lev = max(essais, key=lambda e: (e[0], e[1]))
    c = paired_bootstrap(y_va, p_lev, PRED_VA[K_REF], L2, N_BOOT, SEED)
    detectes = lambda p: int(((y_va == "haineux") & (p == "haineux")).sum())
    alertes = lambda p: int(((y_va == "non_haineux") & (p == "haineux")).sum())
    config_lev = f"+ L-HSAB à l'entraînement ({len(train_lev)} commentaires), C = {C_lev:g}"
    noter("lhsab", "TF-IDF + SVM", config_lev, "macro-F1 T-HSAB (2 classes)", dev=mf_lev, source="section F")
    noter("lhsab_diff", "TF-IDF + SVM", config_lev, "écart avec la référence [IC 95 %] ; p",
          dev=f"{c['diff_macro_f1']:+.3f} [{c['ic_bas']:+.3f} ; {c['ic_haut']:+.3f}] ; p = {c['p_value']:.3f}", source="section F")
    noter("lhsab_det", "TF-IDF + SVM", config_lev, "haineux détectés / fausses alertes (référence entre parenthèses)",
          dev=f"{detectes(p_lev)} / {alertes(p_lev)} ({detectes(PRED_VA[K_REF])} / {alertes(PRED_VA[K_REF])})", source="section F")
    print(f"+ L-HSAB : macro-F1 = {mf_lev:.4f} (référence {REF.macro_f1:.4f}) ; écart {c['diff_macro_f1']:+.3f}, p = {c['p_value']:.3f}")
    print(f"Haineux détectés : {detectes(p_lev)} contre {detectes(PRED_VA[K_REF])} ; fausses alertes : {alertes(p_lev)} contre {alertes(PRED_VA[K_REF])}")

## G. Faux positifs sur insultes et citations

Le jeu de contraste a été construit par l'équipe, dans la feuille `contraste` de `annotations_equipe.xlsx`. Il croise les mêmes insultes en attaque directe avec des usages non haineux : citation, mention, négation, sens méta, sens littéral.

C'est un jeu construit. Il n'est tiré ni de T-HSAB ni de TUNIZI, et il est présenté comme tel.

In [ ]:
CONTRASTE_RES = None
if CONTRASTE is not None:
    cst = CONTRASTE[CONTRASTE.phrase.str.strip() != ""].copy()
    relu = (cst.label.str.strip() != "").to_numpy()
    cst["etiquette"] = np.where(relu, cst.label.str.strip().str.lower(), cst.label_propose.str.strip().str.lower())
    if "type" not in cst:
        cst["type"] = "—"
    print(f"Jeu de contraste : {len(cst)} phrases, dont {int((~relu).sum())} non relue(s) (étiquette proposée utilisée)")
    cst["svm"], cst["score"] = predire_ref(cst.phrase), score_ref(cst.phrase).round(3)
    if LEX:
        cst["lexique"] = [lx.predire(t, LEX) for t in cst.phrase]
    lignes = []
    for type_, g in cst.groupby("type"):
        nh, h = g[g.etiquette == "non_haineux"], g[g.etiquette == "haineux"]
        ligne = {"type": type_, "n": len(g),
                 "faux positifs SVM": f"{int((nh.svm == 'haineux').sum())} / {len(nh)}" if len(nh) else "—",
                 "attaques détectées SVM": f"{int((h.svm == 'haineux').sum())} / {len(h)}" if len(h) else "—"}
        if LEX:
            ligne["faux positifs lexique"] = f"{int((nh.lexique == 'haineux').sum())} / {len(nh)}" if len(nh) else "—"
        lignes.append(ligne)
        noter(f"contraste::{type_}", "TF-IDF + SVM", f"jeu de contraste, usage « {type_} » ({len(g)} phrases)",
              "SVM et lexique sur cet usage", autre=" ; ".join(f"{k} : {v}" for k, v in ligne.items() if k not in ("type", "n")),
              source="section G")
    CONTRASTE_RES = pd.DataFrame(lignes)
    display(CONTRASTE_RES)
    cst.to_csv(OUT / "contraste_predictions.csv", index=False)
    nh, h = cst[cst.etiquette == "non_haineux"], cst[cst.etiquette == "haineux"]
    config_c = f"jeu de contraste construit par l'équipe ({len(cst)} phrases)"
    noter("contraste_fp", "TF-IDF + SVM", config_c, "faux positifs sur citations, mentions, négations, sens littéral",
          autre=f"{int((nh.svm == 'haineux').sum())} / {len(nh)}", source="section G")
    noter("contraste_det", "TF-IDF + SVM", config_c, "attaques directes détectées",
          autre=f"{int((h.svm == 'haineux').sum())} / {len(h)}", source="section G")
    if LEX:
        noter("contraste_fp_lex", "Lexique (règles)", config_c, "faux positifs sur citations, mentions, négations, sens littéral",
              autre=f"{int((nh.lexique == 'haineux').sum())} / {len(nh)}", source="section G")
else:
    print("Jeu de contraste non fourni.")

## H. Analyse de 30 erreurs, sur la validation

Les 30 erreurs sont tirées au hasard (graine 42) parmi les erreurs de validation du modèle de référence. Leurs catégories ont été relues par l'équipe, dans la feuille `erreurs` de `annotations_equipe.xlsx`.

In [ ]:
PISTES = {
    "Haine implicite (religion, politique, déshumanisation)": "Lexique de haine religieuse et politique (règle Bronze)",
    "Insulte dialectale rare ou déformée (lettres répétées, mots collés ou coupés)": "Réduire les lettres répétées ; enrichir le lexique",
    "Mot sensible dans un message neutre": "Ajouter des exemples neutres qui citent des groupes",
    "Message très court": "Signaler les scores proches de 0 à la relecture",
    "Ironie, moquerie ou message mêlé": "Annoter des exemples d'ironie ; relecture humaine",
    "Étiquette discutable": "Relire l'étiquette et l'arbitrer"}
err = (va.assign(reference=y_va, prediction=PRED_VA[K_REF], score_haine=score_ref(va.text_raw).round(3))
         .loc[lambda d: d.reference != d.prediction])
ech = err.sample(n=N_ERREURS, random_state=SEED)
print(f"Erreurs de validation : {len(err)} ({int((err.reference == 'haineux').sum())} faux négatifs, "
      f"{int((err.reference == 'non_haineux').sum())} faux positifs) ; échantillon : {len(ech)} "
      f"({int((ech.reference == 'haineux').sum())} FN, {int((ech.reference == 'non_haineux').sum())} FP)")
noter("erreurs", "TF-IDF + SVM", "erreurs de validation du modèle de référence", "faux négatifs / faux positifs",
      dev=f"{int((err.reference == 'haineux').sum())} / {int((err.reference == 'non_haineux').sum())}", source="section H")
TABLEAU3 = None
if ERREURS is not None:
    e = ERREURS.assign(id=ERREURS["id"].str.strip()).drop_duplicates("id").set_index("id")
    cat = ech["id"].map(e["categorie_validee"].str.strip()).replace("", np.nan)
    non_relues = int(cat.isna().sum())
    if "categorie_proposee" in e:
        cat = cat.fillna(ech["id"].map(e["categorie_proposee"].str.strip()).replace("", np.nan))
    if non_relues:
        print(f"ATTENTION : {non_relues} catégorie(s) non relue(s) : la catégorie proposée est utilisée.")
    piste = ech["id"].map(e["piste"].str.strip()) if "piste" in e else pd.Series("", index=ech.index)
    ech = ech.assign(categorie=cat.fillna("non catégorisé"), piste=piste.fillna(""))
    lignes = []
    for categorie, g in ech.groupby("categorie"):
        pistes = [p for p in g.piste if p] or [PISTES.get(categorie, "")]
        lignes.append({"composant": "TF-IDF + SVM", "catégorie d'erreur": categorie, "nombre (sur 30)": len(g),
                       "exemple court": g.loc[g.text_raw.str.len().idxmin(), "text_raw"][:80], "piste de correction": pistes[0]})
    TABLEAU3 = pd.DataFrame(lignes).sort_values("nombre (sur 30)", ascending=False)
    display(TABLEAU3)
colonnes = ["id", "text_raw", "label", "reference", "prediction", "score_haine"] + (["categorie", "piste"] if "categorie" in ech else [])
ech[colonnes].rename(columns={"text_raw": "entree", "prediction": "sortie"}).to_csv(OUT / "erreurs_30_categorisees.csv", index=False)

## I. Reproduction d'un résultat publié : T-HSAB (Haddad, Mulki et Oueslati, 2019)

D'après la description qu'en donnent d'autres articles, le réglage publié serait : Naive Bayes et SVM, unigrammes et bigrammes, fréquence minimale 2, en 2 et 3 classes. Il est rejoué ici sur **notre** split sans fuite.

Les valeurs publiées doivent être **vérifiées dans le tableau de l'article original** avant d'être saisies dans `PUBLIE`. Les sources secondaires ne concordent pas : l'une cite NB 92,3 en 2 classes et SVM 83,6 ; une autre cite NB 83,6 sans préciser la tâche. Laissez `None` tant que ce n'est pas vérifié.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB

# À RENSEIGNER depuis le tableau des résultats de l'article original ; laisser None si non vérifié
PUBLIE = {("NB", "2 classes"): None, ("SVM", "2 classes"): None, ("NB", "3 classes"): None, ("SVM", "3 classes"): None}
METRIQUE_PUBLIEE = "F-mesure de l'article (préciser : macro, pondérée ou par classe)"


def reproduire(train, evaluation):
    cv = CountVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2).fit(train.text_raw)
    Xa, Xb = cv.transform(train.text_raw), cv.transform(evaluation.text_raw)
    lignes = []
    for tache, cible in [("2 classes", lambda s: s.map(B2)), ("3 classes", lambda s: s)]:
        ya, yb = cible(train.label).to_numpy(), cible(evaluation.label).to_numpy()
        for nom, modele in [("NB", MultinomialNB()), ("SVM", LinearSVC(random_state=SEED, max_iter=20000))]:
            p = modele.fit(Xa, ya).predict(Xb)
            lignes.append({"modèle": nom, "tâche": tache, "macro-F1": f1_score(yb, p, average="macro"),
                           "F1 pondéré": f1_score(yb, p, average="weighted"), "exactitude": float((p == yb).mean()),
                           "publié": PUBLIE[(nom, tache)]})
    return pd.DataFrame(lignes)


REPRO_DEV = reproduire(tr, va)
display(REPRO_DEV.round(3))
for _, r in REPRO_DEV.iterrows():
    noter(f"repro::{r['modèle']}::{r['tâche']}", "Reproduction",
          f"T-HSAB, {r['modèle']}, unigrammes + bigrammes (min_df = 2), {r['tâche']}, notre split",
          "macro-F1 / F1 pondéré", dev=f"{r['macro-F1']:.3f} / {r['F1 pondéré']:.3f}", source="section I",
          autre="publié : à renseigner" if pd.isna(r["publié"]) else f"publié : {r['publié']} ({METRIQUE_PUBLIEE})")

## J. Évaluation unique sur le test gelé

Cette section ne tourne que si `EVALUER_TEST = True`. Chaque exécution est inscrite dans `splits/journal_test.csv`. Après cette étape, on ne change plus rien.

In [ ]:
REPRO_TEST = None
if EVALUER_TEST:
    te = th[th.split == SPLIT_FINAL]
    y_te = te.label.map(B2).to_numpy()
    journal = cfg.path("splits") / "journal_test.csv"
    deja = pd.read_csv(journal) if journal.exists() else pd.DataFrame()
    if len(deja):
        print(f"Attention : le test a déjà été évalué {len(deja)} fois (voir splits/journal_test.csv).")
    preds = {nom: predire_cle(cle, te) for nom, cle in SYSTEMES.items()}
    if LEX:
        preds["Lexique (règles)"] = np.array([lx.predire(t, LEX) for t in te.text_raw])
    lignes = []
    for nom, p in preds.items():
        s = scores(y_te, p, L2)
        lignes.append({"système": nom, **{k: s[k] for k in COLONNES}})
        noter("thsab::lexique" if nom == "Lexique (règles)" else f"thsab::{nom}", test=s["macro_f1"])
    RESULTATS_TEST = pd.DataFrame(lignes)
    RESULTATS_TEST.round(4).to_csv(OUT / "resultats_test.csv", index=False)
    display(RESULTATS_TEST.round(3))
    p_ref = preds[NOM_REF]
    s = scores(y_te, p_ref, L2)
    pt, lo, hi = bootstrap_ci(y_te, p_ref, L2, N_BOOT, SEED)
    noter("thsab::ref_f1h", test=f"{s['f1_haineux']:.3f} / {s['recall_haineux']:.3f}")
    noter("thsab::ref_ic", test=f"[{lo:.3f} ; {hi:.3f}]")
    if LEX:
        noter("thsab::lexique_rappel", test=f"{scores(y_te, preds['Lexique (règles)'], L2)['precision_haineux']:.3f} / "
                                           f"{scores(y_te, preds['Lexique (règles)'], L2)['recall_haineux']:.3f}")
    conf_test = confusion_table(y_te, p_ref, L2)
    conf_test.to_csv(OUT / "confusion_test.csv")
    display(conf_test)
    for nom, p in preds.items():
        if nom != NOM_REF:
            c = paired_bootstrap(y_te, p_ref, p, L2, N_BOOT, SEED)
            noter(f"cmp::{nom}", COMPOSANT.get(nom, "Lexique (règles)"), f"{NOM_REF} − {nom}",
                  "écart de macro-F1 [IC 95 %] ; p (bootstrap apparié)", source="section J",
                  test=f"{c['diff_macro_f1']:+.3f} [{c['ic_bas']:+.3f} ; {c['ic_haut']:+.3f}] ; p = {c['p_value']:.3f}")
    if LEX:
        insulte_nh = (y_te == "non_haineux") & np.array([bool(lx.termes_trouves(t, LEX)) for t in te.text_raw])
        noter("fp_insultes", test=f"{int((p_ref[insulte_nh] == 'haineux').sum())} / {int(insulte_nh.sum())}")
    d = TZ[TUNIZI_FINAL]
    y_tz = d.y.to_numpy()
    p_tz = {nom: predire_ref(d[col]) for nom, col in VUES_TZ.items()}
    for nom, p in p_tz.items():
        noter(f"tunizi::{nom}", test=scores(y_tz, p, L2)["macro_f1"])
    c = paired_bootstrap(y_tz, p_tz["Translittération par règles"], p_tz["Arabizi brut (copie de la source)"], L2, N_BOOT, SEED)
    noter("tunizi::gain", "Translittération (règles)", "règles − Arabizi brut, SVM de référence",
          "écart de macro-F1 [IC 95 %] ; p (bootstrap apparié)", source="section J",
          test=f"{c['diff_macro_f1']:+.3f} [{c['ic_bas']:+.3f} ; {c['ic_haut']:+.3f}] ; p = {c['p_value']:.3f}")
    REPRO_TEST = reproduire(tr, te)
    for _, r in REPRO_TEST.iterrows():
        noter(f"repro::{r['modèle']}::{r['tâche']}", test=f"{r['macro-F1']:.3f} / {r['F1 pondéré']:.3f}")
    entree = {"date": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"), "notebook": "bronze_crosshatetn",
              "configuration": json.dumps({"modele": "svm", "representation": K_REF[1], "texte": K_REF[2],
                                           "C": K_REF[3], "classes": 2}, ensure_ascii=False),
              "macro_f1": round(s["macro_f1"], 4)}
    pd.concat([deja, pd.DataFrame([entree])], ignore_index=True).to_csv(journal, index=False)
    shutil.copy(journal, OUT / "journal_test.csv")
    print(f"Test évalué : macro-F1 de la référence = {s['macro_f1']:.4f} [{lo:.3f} ; {hi:.3f}] ; "
          "journalisé dans splits/journal_test.csv, à versionner.")
else:
    print("Test non évalué (EVALUER_TEST = False) : les colonnes « test » restent vides.")

## K. Latence et sorties de la démo

On mesure la médiane et le 95e centile sur 1 000 appels, pour un message. Le chargement du modèle n'est pas compté.

In [ ]:
def chrono(fonction, n=1000):
    fonction()
    durees = []
    for _ in range(n):
        t0 = time.perf_counter()
        fonction()
        durees.append(time.perf_counter() - t0)
    return 1000 * float(np.median(durees)), 1000 * float(np.percentile(durees, 95))


med_ar, p95_ar = chrono(lambda: score_ref(["يا بهيم سكر فمك"]))
med_az, p95_az = chrono(lambda: score_ref([tl.translitterer("ya bhim sakker fommek")]))
print(f"Arabe : {med_ar:.2f} ms (médiane), {p95_ar:.2f} ms (95e centile)")
print(f"Arabizi avec translittération par règles : {med_az:.2f} ms (médiane), {p95_az:.2f} ms (95e centile)")
noter("latence_ar", "Sortie et démo", "vectorisation + score, 1 message en arabe, 1 000 appels",
      "ms (médiane / 95e centile)", autre=f"{med_ar:.2f} / {p95_ar:.2f}", source="section K")
noter("latence_az", "Sortie et démo", "translittération + vectorisation + score, 1 message en Arabizi, 1 000 appels",
      "ms (médiane / 95e centile)", autre=f"{med_az:.2f} / {p95_az:.2f}", source="section K")
EXEMPLES = ["برافو عليك ربي يوفقك", "يا بهيم سكر فمك", "قالي يا بهيم وضحكنا", "ya bhim sakker fommek"]
demo = pd.DataFrame({"entrée": EXEMPLES, "score": score_ref(EXEMPLES).round(3)})
demo["translittération"] = [tl.translitterer(t) if re.search("[a-z]", t) else "" for t in EXEMPLES]
demo["score après translittération"] = [round(float(score_ref([t])[0]), 3) if t else None for t in demo["translittération"]]
display(demo)
demo.to_csv(OUT / "demo_exemples.csv", index=False)
for _, r in demo.iterrows():
    noter(f"demo::{r['entrée']}", "Sortie et démo", f"« {r['entrée']} »", "score du SVM (> 0 : haineux)",
          autre=f"{r['score']:+.3f}" + (f" ; translittéré : {r['score après translittération']:+.3f}" if r["translittération"] else ""),
          source="section K")

## L. Tableaux 1, 2, 3 et experiments.md

In [ ]:
def fmt(v):
    if v is None:
        return "—"
    if isinstance(v, (float, np.floating)):
        return f"{v:.3f}"
    return str(v)


def val(cle, champ):
    v = RES.get(cle, {}).get(champ)
    if v is None:
        return "non évalué" if champ == "test" and not EVALUER_TEST else "—"
    return fmt(v)


def en_markdown(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lignes = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lignes.append("| " + " | ".join(str(r[c]).replace("|", "/").replace("\n", " ") for c in cols) + " |")
    return "\n".join(lignes)


N_TEST = EFFECTIFS["test"]
t1 = []
def ligne_t1(composant, lot, methode, baseline, metrique, cle_base, cle_bronze, n_test):
    t1.append({"Composant": composant, "Lot": lot, "Méthode Bronze": methode, "Baseline triviale": baseline,
               "Métrique": metrique, "Score baseline": val(cle_base, "test"), "Score Bronze": val(cle_bronze, "test"),
               "Dev (baseline → Bronze)": f"{val(cle_base, 'dev')} → {val(cle_bronze, 'dev')}", "N test": n_test})

if LEX:
    ligne_t1("Lexique (règles)", LOT["Lexique (règles)"], f"lexique relu ({len(LEX)} termes) + préfixes", "majoritaire",
             "macro-F1 (↑)", "thsab::Majoritaire", "thsab::lexique", N_TEST)
ligne_t1("TF-IDF mots + SVM", LOT["TF-IDF + SVM"], RES["thsab::TF-IDF mots + SVM"]["configuration"], "majoritaire",
         "macro-F1 (↑)", "thsab::Majoritaire", "thsab::TF-IDF mots + SVM", N_TEST)
ligne_t1(NOM_REF, LOT["TF-IDF + SVM"], RES[f"thsab::{NOM_REF}"]["configuration"], "majoritaire",
         "macro-F1 (↑)", "thsab::Majoritaire", f"thsab::{NOM_REF}", N_TEST)
for nom in ["Normalisation légère", "Normalisation forte"]:
    if nom in SYSTEMES:
        ligne_t1(nom + " (règles)", LOT["Normalisation (règles)"], RES[f"thsab::{nom}"]["configuration"], "texte brut",
                 "macro-F1 du SVM (↑)", f"thsab::{NOM_BRUT}", f"thsab::{nom}", N_TEST)
ligne_t1("Translittération (règles)", LOT["Translittération (règles)"], "chiffres, digraphes, voyelles brèves",
         "copie de la source (Arabizi brut)", "macro-F1 du SVM sur TUNIZI (↑)",
         "tunizi::Arabizi brut (copie de la source)", "tunizi::Translittération par règles", len(TZ["test"]))
TABLEAU1 = pd.DataFrame(t1)
if KAPPA_RES:
    TABLEAU1 = pd.concat([TABLEAU1, pd.DataFrame([{
        "Composant": "Annotation pilote", "Lot": LOT["Annotation pilote"], "Méthode Bronze": "double annotation aveugle",
        "Baseline triviale": "—", "Métrique": "kappa de Cohen, 2 classes (↑)", "Score baseline": "—",
        "Score Bronze": fmt(KAPPA_RES["kappa_2"]), "Dev (baseline → Bronze)": "—", "N test": KAPPA_RES["n"]}])], ignore_index=True)

TABLEAU2 = pd.DataFrame([{
    "Composant": NOM_REF, "Métrique": "macro-F1 (↑)", "FR": "non couvert", "AR (standard)": "non mesuré",
    "Tunisien (arabe)": f"{val(f'thsab::{NOM_REF}', 'test')} (N = {N_TEST})",
    "Tunisien (arabizi) brut": f"{val('tunizi::Arabizi brut (copie de la source)', 'test')} (N = {len(TZ['test'])})",
    "Tunisien (arabizi) translittéré par règles": f"{val('tunizi::Translittération par règles', 'test')} (N = {len(TZ['test'])})",
    "EN": "non couvert"}])

for nom, table in [("tableau1_composants_test", TABLEAU1), ("tableau2_par_ecriture", TABLEAU2)] + \
                  ([("tableau3_erreurs", TABLEAU3)] if TABLEAU3 is not None else []):
    table.to_csv(OUT / f"{nom}.csv", index=False)
    display(table)

journal_exp = pd.DataFrame([{"date": r["date"], "composant": r["composant"], "configuration": r["configuration"],
                             "métrique": r["métrique"], "dev": fmt(r["dev"]), "test": fmt(r["test"]),
                             "autre": fmt(r["autre"]), "source": r["source"]} for r in RES.values()])
texte = [
    "# experiments.md : CrossHateTN, palier Bronze", "",
    f"Généré par `notebooks/bronze_crosshatetn.ipynb` le {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}.",
    f"Environnement : Python {platform.python_version()}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}.",
    f"Split gelé le {DATE_GEL} ; SHA-256 des identifiants du test : `{HASH_TEST}`.",
    f"Test évalué dans cette exécution : {'oui' if EVALUER_TEST else 'non'} (journal : `splits/journal_test.csv`).",
    "Dev = validation T-HSAB (894 commentaires) et TUNIZI-dev ; test = test gelé T-HSAB (893) et TUNIZI-test.",
    "Tag Git du code exécuté : `bronze` (à créer sur le commit qui contient ce notebook).", "",
    "## Journal des mesures", "", en_markdown(journal_exp), "",
    "## Tableau 1 : résultats par composant sur le test fixé", "", en_markdown(TABLEAU1), "",
    "## Tableau 2 : résultats par écriture", "", en_markdown(TABLEAU2), ""]
if TABLEAU3 is not None:
    texte += ["## Tableau 3 : analyse d'erreurs (30 erreurs de validation)", "", en_markdown(TABLEAU3), ""]
(OUT / "experiments.md").write_text("\n".join(texte), encoding="utf-8")
journal_exp.to_csv(OUT / "experiments.csv", index=False)

archive = WORK / "bronze_outputs.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print("Sorties :", OUT, "| archive :", archive)

## M. Extraits de code pour les diapos (10 lignes au maximum)

In [ ]:
from crosshatetn import splits, text


def extrait(objet, n=10):
    lignes, debut = inspect.getsourcelines(objet)
    try:
        fichier = Path(inspect.getsourcefile(objet)).resolve().relative_to(REPO.resolve())
    except (TypeError, ValueError):
        fichier = "notebooks/bronze_crosshatetn.ipynb"
    print(f"--- {fichier}, lignes {debut} à {debut + min(n, len(lignes)) - 1}")
    print("".join(lignes[:n]))


for objet in [text.normalize_light, text.dedup_key, splits.fresh_assignment, lx.termes_trouves, tl.translitterer_mot,
              representation]:
    try:
        extrait(objet)
    except (OSError, TypeError):
        print(f"--- {objet.__name__} : source non disponible dans cet environnement")

## Où utiliser chaque sortie

| Sortie (`bronze_outputs/`) | Diapo ou pièce à remettre |
|---|---|
| `donnees_*.csv`, preuve du gel (section A) | Données : corpus chiffré, split et gel |
| `double_annotation_tunizi.csv`, kappa (section B) | Données : annotation pilote, kappa et effectif |
| `tableau1_composants_test.csv` | Résultats par composant, avec les extraits de code de la section M |
| `tableau2_par_ecriture.csv` | Scores par écriture : arabe et Arabizi |
| `contraste_predictions.csv` (section G) | Faux positifs sur insultes et citations |
| `tableau3_erreurs.csv`, `erreurs_30_categorisees.csv` | Analyse d'erreurs |
| Section I | Résultat publié reproduit |
| `demo_exemples.csv`, latence (section K) | Démo |
| `experiments.md`, `journal_test.csv` | À versionner dans le dépôt, puis tag `bronze` |